# TriPendulum-8 MJX/JAX PPO 一键训练
GPU 上并行运行 MJX 物理和 JAX PPO。原 SB3 项目不会被覆盖。

In [ ]:
REPO_URL = 'https://github.com/XuanheGuo/TriPendulum-8.git'
REPO_BRANCH = 'mjx-ppo'
PROJECT_DIR = '/content/TriPendulum-8-MJX'
DRIVE_ROOT = '/content/drive/MyDrive/TriPendulum-8-MJX'
RUN_NAME = 'mjx_ppo_curriculum_01'
NUM_TIMESTEPS = 500_000_000  # fallback only; actual per-stage timesteps come from configs/mjx_ppo.yaml
NUM_ENVS = 4096


In [ ]:
REPO_URL = 'https://github.com/XuanheGuo/TriPendulum-8.git'
REPO_BRANCH = 'mjx-ppo'
PROJECT_DIR = '/content/TriPendulum-8-MJX'
DRIVE_ROOT = '/content/drive/MyDrive/TriPendulum-8-MJX'
RUN_NAME = 'mjx_ppo_fresh_01'
NUM_TIMESTEPS = 50_000_000
NUM_ENVS = 4096


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
os.chdir(PROJECT_DIR)


In [ ]:
!pip uninstall -y gym >/dev/null 2>&1 || true
# 先单独装 JAX CUDA（Colab 推荐方式，避免重启后内核崩溃）
!pip install -q "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
# 再装其余依赖（不 force-reinstall）
!pip install -q -r requirements-mjx.txt
import jax, brax, flax, mujoco, mujoco_playground
print('jax:', jax.__version__)
print('brax:', brax.__version__)
print('flax:', flax.__version__)
print('mujoco:', mujoco.__version__)
print('mujoco_playground: imported')
print('backend:', jax.default_backend())
print('devices:', jax.devices())
assert jax.default_backend() == 'gpu', '请在 Runtime -> Change runtime type 中选择 GPU'


In [ ]:
import yaml, pathlib
run_dir = pathlib.Path(DRIVE_ROOT) / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
with open('configs/mjx_ppo.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['ppo']['num_timesteps'] = NUM_TIMESTEPS
cfg['ppo']['num_envs'] = NUM_ENVS
with open('/content/mjx_colab.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('persistent output:', run_dir)


In [ ]:
%load_ext tensorboard
TB_DIR = str(run_dir / 'logs')
%tensorboard --logdir $TB_DIR --reload_interval 30


In [ ]:
!python -m mjx_backend.train_ppo --config /content/mjx_colab.yaml --num-timesteps {NUM_TIMESTEPS} --output-dir '{run_dir}'


In [ ]:
checkpoint = run_dir / 'checkpoints/final.params'
evaluation_file = run_dir / 'evaluation.json'
!python -m mjx_backend.evaluate --config /content/mjx_colab.yaml --checkpoint '{checkpoint}' --episodes 20 --output '{evaluation_file}'
